# Description

This notebook generates a keywords clustering of the studies. The output file is in EPS format.

# Requirements

An environment with:
- matplotlib
- pandas
- openpyxl
- scikit-learn
- sentence-transformers

# Prerequisites

- An excel file ```filename``` containing the references in ```sheet_name``` with at least the column ```column_name```.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

OSError: [WinError 126] Le module spécifié est introuvable. Error loading "c:\Users\maria\Documents\Gitlab\bibMining\.venv\lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [ ]:
filename = '../data/refs.xlsx'
sheet_name = 'RAW'
output_filename = "keywords_clustering.eps"
column_name = 'Keywords'
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"


In [ ]:
data = pd.read_excel(filename, sheet_name=sheet_name)
data = data[~data[column_name].isna()]
df = data.rename(columns={column_name: 'keywords'})

In [ ]:
keywords = []
for kw_list in df["keywords"].dropna():
    for kw in kw_list.split(";"):
        kw = kw.strip().lower()
        if kw: 
            keywords.append(kw)

unique_keywords = list(set(keywords))

model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(unique_keywords, convert_to_numpy=True)  # déjà par défaut True

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)

plot_df = pd.DataFrame({
    "keyword": unique_keywords,
    "x": coords[:, 0],
    "y": coords[:, 1],
    "cluster": labels
})



In [ ]:
plt.figure(figsize=(10,7))

clusters_names = sorted(plot_df["cluster"].unique())

for cluster_id in clusters_names:
    subset = plot_df[plot_df["cluster"] == cluster_id]
    plt.scatter(subset["x"], subset["y"], label=f"Cluster {cluster_id}", alpha=0.6)

    cx, cy = subset["x"].mean(), subset["y"].mean()
    plt.text(cx, cy, f"Cluster {cluster_id}", fontsize=12, weight="bold",
             bbox=dict(facecolor="white", alpha=0.6, edgecolor="grey"))


for _, row in plot_df.iterrows():
    plt.text(row["x"], row["y"], row["keyword"], fontsize=7, alpha=0.8, clip_on=True, va="center")

plt.legend(loc='upper left')
plt.tight_layout()

plt.savefig(output_filename)
plt.show()

In [ ]:

clusters_keywords = {}
for cluster_id in sorted(plot_df["cluster"].unique()):
    clusters_keywords[cluster_id] = plot_df[plot_df["cluster"] == cluster_id]["keyword"].tolist()

for cid, kws in clusters_keywords.items():
    print(f"\n=== Cluster {cid} ===")
    for kw in kws:
        print(f" - {kw}")